# RQ3 — Citation Link Agent Evaluation

**Research Question:**  
> *How accurately can an LLM-based Citation Link Agent extract explicit bibliographic references from academic papers compared with Semantic Scholar reference records?*

## Core Evaluation Standards (Evaluator-Corrected)
This notebook implements the following methodological corrections to ensure mathematical integrity:
1. **Strict One-to-One Matching:** Deduplicates predictions and gold references prior to matching. Once a gold reference is matched to a predicted citation, it is popped from the pool to prevent double-counting.
2. **Pre-Scoring Validation:** Requires every prediction to contain at least one usable identifier (Title, DOI, arXiv ID, or Semantic Scholar ID). Empty or malformed LLM outputs are logged in the audit metadata as invalid but are excluded from TP/FP/FN metrics.
3. **Gold Database Usability Check:** Automatically detects and excludes papers where the Semantic Scholar API returned zero references while the agent extracted multiple valid citations (representing API database holes rather than true zero-citation baselines).
4. **Cache Provenance Verification:** Validates the prompt version, model name, and code hash of `src/agents/link_agent.py` in the prediction cache to ensure experimental reproducibility.
5. **Mathematical Integrity Assertions:** Enforces local and global assertions (e.g., $TP + FN == \text{unique\_gold}$) at runtime, failing clearly if set boundaries are violated.

> [!IMPORTANT]
> **RQ1 and RQ2 are completely frozen.** This notebook does not modify `data/generated_notes`, existing Markdown links, embeddings, retrieval indexes, or any RQ1/RQ2 result files.

## Expected runtime
- **Gold standard build (Cell 2):** ~3 minutes (Semantic Scholar API, 40 papers)
- **Citation Link Agent (Cell 3):** seconds (instantly loads from the validated provenance cache).

## Cell 0 — Install / Verify Dependencies
Run this cell first to ensure the active Jupyter kernel has all required packages.

In [25]:
import subprocess, sys
pkgs = ['google-generativeai', 'python-dotenv', 'requests', 'pandas']
for pkg in pkgs:
    try:
        __import__(pkg.replace('-', '_').split('>=')[0])
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('[OK] All dependencies present.')

Installing google-generativeai...
Installing python-dotenv...
[OK] All dependencies present.


## Cell 1 — Imports and Configuration

In [26]:
import json, re, sys, time, unicodedata, requests, hashlib
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv(Path('.env'), override=False)
except ImportError:
    pass

import os
S2_API_KEY = os.getenv('S2_API_KEY')
assert S2_API_KEY, 'ERROR: S2_API_KEY not found in .env'

# Paths
NOTES_DIR        = Path('data/generated_notes')
EXTRACTED_DIR    = Path('data/extracted_text')
GOLD_PATH        = Path('data/gold_labels/rq3_full_reference_gold.json')
PREDICTIONS_PATH = Path('data/results/rq3_agent_predictions.json')
OUTPUT_PATH      = Path('data/results/rq3_citation_link_evaluation.json')

corpus_ids = sorted(p.stem for p in NOTES_DIR.glob('*.md'))
corpus_set = set(corpus_ids)
assert len(corpus_ids) == 40, f'Expected 40 corpus papers, found {len(corpus_ids)}'
print(f'Corpus: {len(corpus_ids)} papers loaded.')
print(f'S2_API_KEY: present ({len(S2_API_KEY)} chars)')

Corpus: 40 papers loaded.
S2_API_KEY: present (44 chars)


## Cell 2 — Build Gold Standard (Semantic Scholar API)

Queries the full outgoing reference list for each of the 40 corpus papers.  
**This cell takes ~3 minutes.** All 40 papers are resolved and paginated completely.

References are NOT restricted to the 40-paper corpus — all external references remain in the gold set.

*Academic basis: Lo et al. (2020) S2ORC; Kinney et al. (2023) Semantic Scholar Open Data Platform.*

In [27]:
S2_HEADERS = {'x-api-key': S2_API_KEY}
BASE_URL   = 'https://api.semanticscholar.org/graph/v1/paper'
FIELDS     = 'citedPaper.paperId,citedPaper.externalIds,citedPaper.title'
LIMIT, DELAY, MAX_RETRIES = 1000, 1.1, 3

def resolve_s2_id(arxiv_id):
    url = f'{BASE_URL}/arXiv:{arxiv_id}?fields=paperId'
    for _ in range(MAX_RETRIES):
        r = requests.get(url, headers=S2_HEADERS, timeout=15)
        if r.status_code == 200: return r.json().get('paperId')
        if r.status_code == 429: time.sleep(60)
        elif r.status_code == 404: return None
        else: break
    return None

def fetch_refs(s2_id, arxiv_id):
    all_refs, offset = [], 0
    while True:
        url = f'{BASE_URL}/{s2_id}/references?fields={FIELDS}&limit={LIMIT}&offset={offset}'
        for _ in range(MAX_RETRIES):
            r = requests.get(url, headers=S2_HEADERS, timeout=30)
            if r.status_code == 200: data = r.json().get('data', []); break
            if r.status_code == 429: time.sleep(60)
            else: data = []; break
        else: data = []
        if not data: break
        for e in data:
            cp  = e.get('citedPaper', {})
            ext = cp.get('externalIds') or {}
            all_refs.append({
                's2_paper_id': cp.get('paperId'),
                'title':       cp.get('title'),
                'doi':         ext.get('DOI'),
                'arxiv_id':    ext.get('ArXiv'),
                'in_local_corpus': bool(ext.get('ArXiv') and ext['ArXiv'] in corpus_set),
            })
        offset += LIMIT
        time.sleep(DELAY)
        if len(data) < LIMIT: break
    return all_refs

per_paper_gold = {}
failed_gold    = []
total_gold_refs = 0

for i, arxiv_id in enumerate(corpus_ids, 1):
    print(f'[{i:02d}/40] {arxiv_id}', end=' ... ')
    time.sleep(DELAY)
    s2_id = resolve_s2_id(arxiv_id)
    if not s2_id:
        print('FAILED')
        failed_gold.append(arxiv_id)
        per_paper_gold[arxiv_id] = {'s2_paper_id': None, 'references': [], 'failed': True}
        continue
    refs = fetch_refs(s2_id, arxiv_id)
    total_gold_refs += len(refs)
    per_paper_gold[arxiv_id] = {'s2_paper_id': s2_id, 'references': refs, 'failed': False}
    print(f'{len(refs)} refs')

gold_timestamp = datetime.now(timezone.utc).isoformat()
gold_output = {
    'audit_summary': {
        'api_query_timestamp':   gold_timestamp,
        'corpus_papers':         len(corpus_ids),
        'successfully_resolved': len(corpus_ids) - len(failed_gold),
        'failed_resolutions':    len(failed_gold),
        'failed_arxiv_ids':      failed_gold,
        'total_references_fetched': total_gold_refs,
    },
    'per_paper_references': per_paper_gold,
}
GOLD_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(GOLD_PATH, 'w', encoding='utf-8') as f:
    json.dump(gold_output, f, indent=2, ensure_ascii=False)

print(f'\n[OK] Gold standard saved: {len(corpus_ids)-len(failed_gold)}/40 resolved, {total_gold_refs} total references.')
if failed_gold: print(f'Failed: {failed_gold}')

[01/40] 1711.11157 ... 65 refs
[02/40] 2005.11401 ... 67 refs
[03/40] 2007.01282 ... 36 refs
[04/40] 2012.13635 ... 81 refs
[05/40] 2210.06726 ... 54 refs
[06/40] 2212.12393 ... 64 refs
[07/40] 2305.10250 ... 19 refs
[08/40] 2305.19951 ... FAILED
[09/40] 2306.07516 ... 32 refs
[10/40] 2309.15217 ... 30 refs
[11/40] 2310.08560 ... 42 refs
[12/40] 2310.11511 ... 57 refs
[13/40] 2310.17451 ... 58 refs
[14/40] 2312.04474 ... 49 refs
[15/40] 2401.05224 ... 49 refs
[16/40] 2401.15884 ... 42 refs
[17/40] 2401.18059 ... 61 refs
[18/40] 2402.12240 ... 90 refs
[19/40] 2402.14207 ... 65 refs
[20/40] 2402.17753 ... 71 refs
[21/40] 2404.16130 ... 57 refs
[22/40] 2405.14831 ... 97 refs
[23/40] 2406.14177 ... 34 refs
[24/40] 2409.05591 ... 53 refs
[25/40] 2410.02143 ... 34 refs
[26/40] 2410.05779 ... 21 refs
[27/40] 2412.06014 ... 90 refs
[28/40] 2502.01341 ... 54 refs
[29/40] 2502.03274 ... 36 refs
[30/40] 2502.12110 ... 50 refs
[31/40] 2505.13138 ... 89 refs
[32/40] 2505.23061 ... 44 refs
[33/40] 2

## Cell 3 — Citation Link Agent (Cache-Provenanced)

Loads the generated notes predictions. If the cache exists and passes code-hash provenance validation, it loads instantly to protect your API quota. Otherwise, it executes the LLM pipeline.

*Academic basis: Tkaczyk et al. (2015) CERMINE; Councill, Giles & Kan (2008) ParsCit.*

In [28]:
sys.path.insert(0, '.')
from src.agents.link_agent import extract_citation_links, _norm_title

# Build corpus title index for local resolution
corpus_title_index = {}
for note_path in NOTES_DIR.glob('*.md'):
    content = note_path.read_text(encoding='utf-8')
    m = re.search(r'^title:\s*["\']?(.*?)["\']?\s*$', content, re.MULTILINE)
    if m:
        norm = _norm_title(m.group(1).strip())
        if norm: corpus_title_index[norm] = note_path.stem
print(f'Corpus title index: {len(corpus_title_index)} entries')

agent_predictions = {}
if PREDICTIONS_PATH.exists():
    try:
        with open(PREDICTIONS_PATH, encoding='utf-8') as f:
            cached_data = json.load(f)
        prov = cached_data.get('provenance_metadata', {})
        cached = cached_data.get('predictions', {})
        
        # Verify agent code hash
        agent_code = Path('src/agents/link_agent.py').read_text(encoding='utf-8')
        expected_hash = hashlib.sha256(agent_code.encode('utf-8')).hexdigest()
        
        if (prov.get('agent_code_hash') == expected_hash and 
            prov.get('model_name') == 'gemini-flash-latest' and
            prov.get('temperature') == 0.0 and
            prov.get('prompt_version') == 'v1' and
            all(cid in cached for cid in corpus_ids) and len(cached) == 40):
            agent_predictions = cached
            print('[OK] Loaded predictions from cache (rq3_agent_predictions.json). Skipping LLM API calls!')
        else:
            print('Provenance check failed. Re-running LLM API calls...')
    except Exception as e:
        print(f'Cache loading failed: {e}. Running API calls...')

if not agent_predictions:
    print('No complete predictions cache found. Running Citation Link Agent on 40 papers...')
    for i, arxiv_id in enumerate(corpus_ids, 1):
        ext_path = EXTRACTED_DIR / f'{arxiv_id}.json'
        if not ext_path.exists():
            print(f'[{i:02d}/40] {arxiv_id} — no extracted text, skipping')
            agent_predictions[arxiv_id] = {'references': [], 'error': 'no_extracted_text', 'section_fallback': False}
            continue
        doc       = json.load(open(ext_path, encoding='utf-8'))
        full_text = doc.get('text', '')
        if not full_text.strip():
            print(f'[{i:02d}/40] {arxiv_id} — empty text, skipping')
            agent_predictions[arxiv_id] = {'references': [], 'error': 'empty_text', 'section_fallback': False}
            continue
        result = extract_citation_links(full_text, corpus_index=corpus_title_index)
        refs   = result.get('references', [])
        local  = sum(1 for r in refs if r.get('local_arxiv_id'))
        fb     = '[FALLBACK] ' if result.get('section_fallback') else ''
        print(f'[{i:02d}/40] {arxiv_id} — {fb}{len(refs)} refs ({local} local)')
        agent_predictions[arxiv_id] = result
        time.sleep(1.0)

    # Re-save with provenance metadata
    agent_code = Path('src/agents/link_agent.py').read_text(encoding='utf-8')
    code_hash = hashlib.sha256(agent_code.encode('utf-8')).hexdigest()
    wrapped = {
        'provenance_metadata': {
            'model_name': 'gemini-flash-latest',
            'temperature': 0.0,
            'prompt_version': 'v1',
            'agent_code_hash': code_hash,
            'creation_timestamp': datetime.now(timezone.utc).isoformat()
        },
        'predictions': agent_predictions
    }
    PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(PREDICTIONS_PATH, 'w', encoding='utf-8') as f:
        json.dump(wrapped, f, indent=2, ensure_ascii=False)
    print('[OK] Agent predictions saved with provenance.')

    total_preds = sum(len(v.get('references', [])) for v in agent_predictions.values())
    print(f'Total references extracted: {total_preds}')


Corpus title index: 40 entries
Provenance check failed. Re-running LLM API calls...
No complete predictions cache found. Running Citation Link Agent on 40 papers...
[01/40] 1711.11157 — 58 refs (0 local)
[02/40] 2005.11401 — 38 refs (0 local)
[03/40] 2007.01282 — 34 refs (1 local)
[04/40] 2012.13635 — 52 refs (0 local)

[Transient Error] Invalid operation: The `response.text` quick accessor requires the response to contain a valid `Part`, but none were returned. The candidate's [finish_reason](https://ai.google.dev/api/generate-content#finishreason) is 4. Meaning that the model was reciting from copyrighted material.. Sleeping 10s...

[Transient Error] Invalid operation: The `response.text` quick accessor requires the response to contain a valid `Part`, but none were returned. The candidate's [finish_reason](https://ai.google.dev/api/generate-content#finishreason) is 4. Meaning that the model was reciting from copyrighted material.. Sleeping 10s...

[Transient Error] Invalid operation:

## Cell 4 — Strict Transitive Deduplication & Evaluation

This cell implements transitive deduplication, status flag filters (excluding provider blocks and gold database failures), and sequential one-to-one matching.

In [29]:
def norm_arxiv(v):
    if not v: return None
    return re.sub(r'v\d+$', '', str(v).strip().lower())
def norm_doi(v): return str(v).strip().lower() if v else None
def norm_s2(v):  return str(v).strip() if v else None
def norm_t(v):
    if not v: return None
    t = unicodedata.normalize('NFKD', str(v).lower())
    t = re.sub(r'[^\w\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def jaccard(a, b):
    if not a or not b: return 0.0
    sa, sb = set(a.split()), set(b.split())
    return len(sa & sb) / len(sa | sb) if (sa or sb) else 0.0

def fuzzy_cands(agent_ref, gold_refs, thr=0.55):
    at = norm_t(agent_ref.get('title'))
    if not at: return []
    c = [{'gold_title': r.get('title'), 'score': round(jaccard(at, norm_t(r.get('title')) or ''), 4)}
         for r in gold_refs if jaccard(at, norm_t(r.get('title')) or '') >= thr]
    return sorted(c, key=lambda x: -x['score'])[:5]

def deduplicate_references(refs):
    unique_refs = []
    invalid_count = 0
    for ref in refs:
        s2, doi, arx, tit = norm_s2(ref.get('s2_paper_id')), norm_doi(ref.get('doi')), norm_arxiv(ref.get('arxiv_id')), norm_t(ref.get('title'))
        if not any([s2, doi, arx, tit]):
            invalid_count += 1
            continue
        is_duplicate = False
        for u in unique_refs:
            if s2 and norm_s2(u.get('s2_paper_id')) == s2: is_duplicate = True; break
            if doi and norm_doi(u.get('doi')) == doi: is_duplicate = True; break
            if arx and norm_arxiv(u.get('arxiv_id')) == arx: is_duplicate = True; break
            if tit and norm_t(u.get('title')) == tit: is_duplicate = True; break
        if not is_duplicate:
            unique_refs.append(ref)
    return unique_refs, invalid_count

global_tp = global_fp = global_fn = 0
match_counts = {'s2_id': 0, 'doi': 0, 'arxiv_id': 0, 'exact_title': 0}
per_paper_results = {}
invalid_outputs_count = 0
excluded_gold_unusable = []
excluded_provider_blocks = []
per_paper_totals_sum = {'tp': 0, 'fp': 0, 'fn': 0}

for arxiv_id in corpus_ids:
    gold_paper = per_paper_gold.get(arxiv_id, {})
    gold_refs  = gold_paper.get('references', [])
    agent_data = agent_predictions.get(arxiv_id, {})
    agent_refs = agent_data.get('references', [])
    agent_status = agent_data.get('status', 'success')
    
    if gold_paper.get('failed'):
        per_paper_results[arxiv_id] = {'status': 'gold_unusable', 'tp': 0, 'fp': 0, 'fn': 0}
        continue
        
    # 1. Deduplicate references transitively
    unique_gold, _ = deduplicate_references(gold_refs)
    unique_preds, invalid_cnt = deduplicate_references(agent_refs)
    invalid_outputs_count += invalid_cnt
    
    # 2. Exclude gold unusable (API failures returning 0 refs when agent found refs)
    if len(unique_gold) == 0 and len(unique_preds) > 0:
        excluded_gold_unusable.append(arxiv_id)
        per_paper_results[arxiv_id] = {'status': 'gold_unusable', 'tp': 0, 'fp': 0, 'fn': 0}
        continue
        
    # 3. Exclude provider blocks from evaluation scope
    if agent_status == 'provider_block':
        excluded_provider_blocks.append(arxiv_id)
        per_paper_results[arxiv_id] = {'status': 'provider_block', 'tp': 0, 'fp': 0, 'fn': 0}
        continue
        
    # 4. Strict One-to-One Matching
    gold_pool = list(unique_gold)
    matched_gold_indices = set()
    paper_tp = paper_fp = 0
    records = []
    
    for ref in unique_preds:
        match_found = False
        
        s2 = norm_s2(ref.get('s2_paper_id'))
        if s2:
            for idx, g in enumerate(gold_pool):
                if idx not in matched_gold_indices and norm_s2(g.get('s2_paper_id')) == s2:
                    matched_gold_indices.add(idx)
                    match_counts['s2_id'] = match_counts.get('s2_id', 0) + 1
                    paper_tp += 1
                    records.append({'agent_title': ref.get('title'), 'matched': g.get('title'), 'match_type': 's2_id', 'result': 'TP'})
                    match_found = True
                    break
        if match_found: continue
        
        doi = norm_doi(ref.get('doi'))
        if doi:
            for idx, g in enumerate(gold_pool):
                if idx not in matched_gold_indices and norm_doi(g.get('doi')) == doi:
                    matched_gold_indices.add(idx)
                    match_counts['doi'] = match_counts.get('doi', 0) + 1
                    paper_tp += 1
                    records.append({'agent_title': ref.get('title'), 'matched': g.get('title'), 'match_type': 'doi', 'result': 'TP'})
                    match_found = True
                    break
        if match_found: continue
        
        arx = norm_arxiv(ref.get('arxiv_id'))
        if arx:
            for idx, g in enumerate(gold_pool):
                if idx not in matched_gold_indices and norm_arxiv(g.get('arxiv_id')) == arx:
                    matched_gold_indices.add(idx)
                    match_counts['arxiv_id'] = match_counts.get('arxiv_id', 0) + 1
                    paper_tp += 1
                    records.append({'agent_title': ref.get('title'), 'matched': g.get('title'), 'match_type': 'arxiv_id', 'result': 'TP'})
                    match_found = True
                    break
        if match_found: continue
        
        tit = norm_t(ref.get('title'))
        if tit:
            for idx, g in enumerate(gold_pool):
                if idx not in matched_gold_indices and norm_t(g.get('title')) == tit:
                    matched_gold_indices.add(idx)
                    match_counts['exact_title'] = match_counts.get('exact_title', 0) + 1
                    paper_tp += 1
                    records.append({'agent_title': ref.get('title'), 'matched': g.get('title'), 'match_type': 'exact_title', 'result': 'TP'})
                    match_found = True
                    break
        if match_found: continue
        
        paper_fp += 1
        records.append({'agent_title': ref.get('title'), 'result': 'FP', 'fuzzy_match_candidates': fuzzy_cands(ref, gold_pool)})
        
    paper_fn = len(gold_pool) - len(matched_gold_indices)
    fn_list = [gold_pool[i] for i in range(len(gold_pool)) if i not in matched_gold_indices]
    
    # Assert local paper level integrity
    assert paper_tp + paper_fn == len(unique_gold), f'Math discrepancy in {arxiv_id}'
    assert paper_tp + paper_fp == len(unique_preds), f'Math discrepancy in {arxiv_id}'
    
    global_tp += paper_tp
    global_fp += paper_fp
    global_fn += paper_fn
    
    per_paper_totals_sum['tp'] += paper_tp
    per_paper_totals_sum['fp'] += paper_fp
    per_paper_totals_sum['fn'] += paper_fn
    
    per_paper_results[arxiv_id] = {
        'status': agent_status, 'tp': paper_tp, 'fp': paper_fp, 'fn': paper_fn,
        'gold_refs': unique_gold, 'preds': unique_preds, 'agent_records': records, 'false_negatives': fn_list
    }

# Assert global totals sum matching
scored_papers = [p for p in corpus_ids if per_paper_results[p]['status'] not in ['gold_unusable', 'provider_block']]
assert global_tp == per_paper_totals_sum['tp'], 'Global TP mismatch'
assert global_fp == per_paper_totals_sum['fp'], 'Global FP mismatch'
assert global_fn == per_paper_totals_sum['fn'], 'Global FN mismatch'

precision = global_tp / (global_tp + global_fp) if (global_tp + global_fp) > 0 else 0.0
recall    = global_tp / (global_tp + global_fn) if (global_tp + global_fn) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print('='*60)
print('RQ3 — CITATION LINK AGENT RESULTS (CORRECTED CORPUS)')
print('='*60)
df = pd.DataFrame([
    {'Metric': 'True Positives (TP)',  'Value': global_tp},
    {'Metric': 'False Positives (FP)', 'Value': global_fp},
    {'Metric': 'False Negatives (FN)', 'Value': global_fn},
    {'Metric': 'Precision',            'Value': f'{precision:.4f}'},
    {'Metric': 'Recall',               'Value': f'{recall:.4f}'},
    {'Metric': 'F1-Score',             'Value': f'{f1:.4f}'},
])
print(df.to_string(index=False))
print(f'\nMatch type breakdown: {match_counts}')
print(f'Invalid agent outputs filtered: {invalid_outputs_count}')
print(f'Gold unusable papers excluded (API failures): {excluded_gold_unusable}')
print(f'Provider blocked papers excluded: {excluded_provider_blocks}')
print(f'Scored papers count: {len(scored_papers)}/40')

RQ3 — CITATION LINK AGENT RESULTS (CORRECTED CORPUS)
              Metric  Value
 True Positives (TP)   1060
False Positives (FP)    197
False Negatives (FN)    988
           Precision 0.8433
              Recall 0.5176
            F1-Score 0.6415

Match type breakdown: {'s2_id': 0, 'doi': 63, 'arxiv_id': 300, 'exact_title': 697}
Invalid agent outputs filtered: 0
Gold unusable papers excluded (API failures): ['2510.14538', '2602.23878']
Provider blocked papers excluded: []
Scored papers count: 37/40


## Cell 5 — Save Final Results

In [30]:
final_output = {
    'audit_metadata': {
        'evaluation_timestamp': datetime.now(timezone.utc).isoformat(),
        'research_question':    'How accurately can an LLM-based Citation Link Agent extract explicit bibliographic references from academic papers compared with Semantic Scholar reference records?',
        'gold_build_timestamp': gold_timestamp,
        'gold_papers_resolved': len(corpus_ids) - len(failed_gold),
        'gold_total_references': total_gold_refs,
        'rq1_rq2_frozen': True,
        'one_to_one_matching': True,
        'transitive_deduplication': True,
        'invalid_agent_outputs_filtered': invalid_outputs_count,
        'excluded_gold_unusable_papers': excluded_gold_unusable,
        'excluded_provider_blocked_papers': excluded_provider_blocks,
        'scored_papers_count': len(scored_papers),
    },
    'overall_metrics': {
        'true_positives':  global_tp,
        'false_positives': global_fp,
        'false_negatives': global_fn,
        'precision':       round(precision, 5),
        'recall':          round(recall, 5),
        'f1':              round(f1, 5),
        'match_type_breakdown': match_counts,
    },
    'per_paper_results': per_paper_results,
}
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)
print(f'[OK] Results saved to {OUTPUT_PATH}')

[OK] Results saved to data\results\rq3_citation_link_evaluation.json
